### Importing Libraries 

In [ ]:
import websocket 
import json 
import pandas as pd
import time

In [2]:
SYMBOL="btcusdt"
STREAM_URL=f"wss://stream.binance.com:9443/ws/{SYMBOL}@bookTicker"
MAX_TICKS=5000 # Number of rows to collect 

# Global list to store our processed data
tick_data=[]
start_time=time.time()


In [ ]:
def on_message(ws,message):
    """Triggered every time Binance sends a new tick"""
    data= json.loads(message)

    # Extract raw level 2 Order Book Data 
    bid_price= float(data['b'])
    bid_qty=float(data['B'])
    ask_price= float(data['a'])
    ask_qty=float(data["A"])


    # --- Feature Engineering (On the fly) ---
    # 1.Mid-Price: The exact Center of the spread 
    mid_price= (bid_price + ask_price )/2.0

    # Order Book Imbalance (OBI): Pressure between buyers and sellers
    # Range is [-1 to 1]. Positive means more buying pressure, negative means selling pressure.
     
    obi = (bid_qty - ask_qty) / (bid_qty + ask_qty)

    tick_data.append({
        "timestamp":time.time(),
        "mid_price": mid_price,
        "obi": obi,
        "bid_qty": bid_qty,
        "ask_qty": ask_qty
        })
    
    # progress tracker
    if len(tick_data) % 500==0:
        print(f"collect {len(tick_data)}/{MAX_TICKS} ticks...")
    # stop condition
    if len(tick_data) >= MAX_TICKS:
        print(f"\n Target reached in {round(time.tme() - start_time, 2 )} seconds. Closing connection...")

        ws.close()
    
def on_error(ws,error):
    print(f"Error: {error}")

def on_close(ws,close_status_code, close_msg):
    print("Connection closed.")

    df= pd.DataFrame(tick_data)
    df.to_csv("btc_hft_ticks.csv", index=False)
    print(f"Successfully saved {len(df)}")